## Setup

In [0]:
from pyspark.sql import SparkSession
def create_spark_session(app_name: str = "CineClubWrapped") -> SparkSession:

    return (
        SparkSession.builder
        .appName(app_name)
        # .config(...)  # cluster / warehouse configs if needed
        .getOrCreate()
    )


In [0]:
from pyspark.sql import functions as F

T_trans = spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")
T_sessions = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_session")
T_locations = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_location")
T_films = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_film")
T_ticket = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_ticket_type")
T_date = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_date")
T_items = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_item")
T_genre = spark.table("cpx_dataanalytics_gold.customer360.film_composite_genres")
T_subscriber = spark.table("cpx_dataanalytics_gold.marketing.subscriber_profile_snapshot")
T_events = spark.table("cpx_dataanalytics_gold.marketing.subscription_events")

In [0]:
today = F.current_date()
yesterday = F.date_sub(F.current_date(), 1)

df_pre = (
    T_trans
    .join(T_locations, T_trans.LocationId == T_locations.LocationID, "left")
    .join(T_ticket, T_trans.TicketTypeSkey == T_ticket.tickettypeskey, "left")
    .join(T_date.select("dateid", "datefull"), T_trans.VisitDateId == T_date.dateid, "left")
    .filter(
        (T_locations.LC_Location_Type_Description == "Cineplex Theatre")
        & (F.col("STOREFLAG") == "No")
        & (F.col("RecRoomFLAG") == "No")
        & (T_trans.CDE_ID.isNotNull())
        & (F.col("datefull").between("2025-01-01", yesterday))
        & ((F.col("TicketFLAG") == "Yes") | (F.col("ConcessionFLAG") == "Yes") | (F.col("ticket_type_id") == "1357") )
        
    )
)


df_pre2 = (
    T_trans
    .join(T_locations, T_trans.LocationId == T_locations.LocationID, "left")
    .join(T_ticket, T_trans.TicketTypeSkey == T_ticket.tickettypeskey, "left")
    .join(T_date.select("dateid", "datefull"), T_trans.VisitDateId == T_date.dateid, "left")
    .filter(
        ((F.col("CineClubFLAG") == "Yes") | (F.col("ticket_type_id") == "1357"))
        & (T_trans.CDE_ID.isNotNull())
        & (F.col("datefull").between("2025-01-01", yesterday))
        & ((F.col("TicketFLAG") == "Yes") | (F.col("ConcessionFLAG") == "Yes"))
    ).dropDuplicates()
)

count_members_for_cumulative = (T_trans                                                                                              #Total Number of distinct CDE_IDs who made a cineclub purchase this year
                                 .join(T_date, T_trans.VisitDateId == T_date.dateid, "left")
                                 .filter((F.col("CineClubFLAG") == 'Yes') & (F.col("datefull").between("2025-01-01", yesterday)))
                                .select(F.countDistinct("CDE_ID").alias("Distinct_Members"))
                                .collect()[0]["Distinct_Members"]
)
# display(count_members_for_cumulative)

In [0]:
from pyspark.sql import functions as F, Window

# ---------------------------------------------
# 1. Take only ticket rows (purchases + refunds)
# ---------------------------------------------
df_tickets = (
    df_pre
    .filter(F.col("TicketFLAG") == "Yes")
    .select(
        "record_id",            # unique row identifier
        "CDE_ID",
        "film_id",
        "datefull",
        "Transaction_DateTime", # used to match refund → purchase
        "Quantity"
    )
)

# ---------------------------------------------
# 2. Split into purchases (+1) and refunds (-1)
# ---------------------------------------------
df_purchases = df_tickets.filter(F.col("Quantity") > 0)
df_refunds   = df_tickets.filter(F.col("Quantity") < 0)

# ---------------------------------------------
# 3. For each refund, find the closest earlier purchase
#    (same customer + same film + purchased before refund)
# ---------------------------------------------
w = (
    Window.partitionBy("r.record_id")
          .orderBy(F.col("p.Transaction_DateTime").desc())  # nearest purchase first
)

df_matches = (
    df_refunds.alias("r")
    .join(
        df_purchases.alias("p"),
        (
            (F.col("r.CDE_ID") == F.col("p.CDE_ID")) &      # same customer
            (F.col("r.film_id") == F.col("p.film_id")) &    # same film
            (F.col("p.Transaction_DateTime") <=             # purchase happened
             F.col("r.Transaction_DateTime"))               # before refund
        ),
        "left"
    )
    .withColumn("rn", F.row_number().over(w))               # pick nearest match
    .filter(F.col("rn") == 1)                               
    .select(
        F.col("p.record_id").alias("purchase_record_id"),
        F.col("r.record_id").alias("refund_record_id")
    )
)

# ---------------------------------------------
# 4. Mark both purchase + refund rows for removal
# ---------------------------------------------
df_cancelled_ids = (
    df_matches
    .select("purchase_record_id", "refund_record_id")
    .select(F.explode(F.array("purchase_record_id", "refund_record_id"))
            .alias("record_id"))
    .distinct()
)

# ---------------------------------------------
# 5. Remove cancelled transactions from full table
# ---------------------------------------------
df_all_trans = (
    df_pre
    .join(df_cancelled_ids, on="record_id", how="left_anti")
    .withColumnRenamed("SessionSKey", "sessionskey")
)


In [0]:
# Concession items cannot be refunded. Hence we dont add any refund logic.

df_pre_conc = (
    T_trans
    .join(T_items, T_trans.ItemSkey == T_items.itemskey, "left")
    .join(T_locations, T_trans.LocationId == T_locations.LocationID, "left")
    .join(T_date, T_trans.VisitDateId == T_date.dateid, "left")
    .filter(
        (F.col("ConcessionFLAG") == "Yes") &
        (T_locations.LC_Location_Type_Description == "Cineplex Theatre") &
        (F.col("STOREFLAG") == "No") &
        (F.col("RecRoomFLAG") == "No") &
        (F.col("CDE_ID").isNotNull()) &
        (F.col("datefull").between("2025-01-01", yesterday))
    )
    .select("CDE_ID", "datefull", "Item_Description", "Quantity", T_trans.LocationId, "Item_ID")
)


df_pre_conc_test = (
    T_trans
    .join(T_items, T_trans.ItemSkey == T_items.itemskey, "left")
    .join(T_locations, T_trans.LocationId == T_locations.LocationID, "left")
    .join(T_date, T_trans.VisitDateId == T_date.dateid, "left")
    .filter(
        (T_locations.LC_Location_Type_Description == "Cineplex Theatre") &
        (F.col("STOREFLAG") == "No") &
        (F.col("RecRoomFLAG") == "No") &
        (F.col("datefull").between("2025-01-01", yesterday))
    )
    .select("CDE_ID", "datefull", "Item_Description", "Quantity", T_trans.LocationId, "Item_ID")
)

In [0]:
df_cineclub_members = (
    T_subscriber
    .filter(
           (F.col("snapshot_date_id") == "20251202") &
           (F.col("plan_id").isin("annual1", "gift-plan")) &
           (F.col("renewal_status").isin("Active", "Active_Non_Renewing", "Active", "Active_Annnual_Upgrade"))
    )
    .select("cde_id", "renewal_status", "recognitions_left", "plan_id", "cc_user_profile_id" )
    .withColumnRenamed("cde_id", "CDE_ID")
    .filter(F.col("recognitions_left") < 54)
)

df_cineclub_members_2 = (
    T_subscriber
    .filter(
           (F.col("snapshot_date_id") == "20251202") 
    )
    .select("cde_id", "renewal_status", "recognitions_left", "plan_id", "cc_user_profile_id" )
    .withColumnRenamed("cde_id", "CDE_ID")
    .filter(F.col("recognitions_left") < 54)
)

# display(df_cineclub_members.count())

In [0]:
df_ticket_visits = (
    df_all_trans
    .filter(F.col("TicketFLAG") == "Yes")
    # .filter(F.col("CineClubFLAG") == "Yes")
    .select("CDE_ID", "datefull", T_trans.LocationId, "Quantity")
    .groupBy("CDE_ID", "datefull", T_trans.LocationId)
    .agg(
        F.sum(F.col("Quantity")).alias("No_of_Tickets")
    )
    .filter(F.col("No_of_Tickets")>0)
    .drop("No_of_Tickets")
    .dropDuplicates()   
)


df_concession_visits = (
    df_pre_conc
    .select("CDE_ID", "datefull", T_trans.LocationId)
    .distinct()
)

df_visits = (                           #ALl VISITS INCLUDING CONCESSION VISITS
    df_ticket_visits
    .unionByName(df_concession_visits)
    .distinct()
)


# Modules

### SCENE Points Calculation
This is a CDE_ID Level Aggregation. Directly Left Join with the Main Table.

In [0]:
#This calculates the Total Scene Points Earned by All CineClub Customers in the current year. The aggregation directly takes Refunds into account.

df_scene_table = (
    T_trans
    .join(T_locations, T_trans.LocationId == T_locations.LocationID, "left")
    .join(T_ticket, T_trans.TicketTypeSkey == T_ticket.tickettypeskey, "left")
    .join(T_date.select("dateid", "datefull"), T_trans.VisitDateId == T_date.dateid, "left")
    .filter(
        (T_locations.LC_Location_Type_Description == "Cineplex Theatre")
        & (F.col("STOREFLAG") == "No")
        & (F.col("RecRoomFLAG") == "No")
        & (T_trans.CDE_ID.isNotNull())
        & (F.col("datefull").between("2025-01-01", yesterday))
        & ((F.col("TicketFLAG") == "Yes") | (F.col("ConcessionFLAG") == "Yes") | (F.col("ticket_type_id") == "1357") | F.col("Box_Office_SundryFLAG") == "Yes")
        
    )
)



df_scene_points = (df_scene_table
                   .groupBy("CDE_ID")
                   .agg(
                       F.sum("SCENE_Points_Earned").alias("Total_Scene_Points"))
                   )

# Total_scene_points = df_scene_points.agg(F.sum(F.coalesce("Total_Scene_Points", F.lit(0)))).first()[0]

In [0]:
display(df_scene_points.filter(F.col("CDE_ID")==11509008))

### Total Visits, Top 10% Visitors and Concession Badge Per CDE_ID
This is a CDE_ID Level Aggregation. Directly Left Join with the Main Table. **Only the Variable df_visit_badges**.

In [0]:
from pyspark.sql.window import Window
import math

# -----------------------------------------------------------
# Aggregate total visits per member
# -----------------------------------------------------------
df_total_visits = (
    df_visits           #ALl VISITS INCLUDING CONCESSION VISITS
    .groupBy("CDE_ID")
    .agg(F.count("*").alias("Total_Visits"))
)

# -----------------------------------------------------------
# Top 10% Visit Badge
# -----------------------------------------------------------
n_members = df_total_visits.count()
top_n = max(1, math.floor(n_members * 0.10))

w = Window.orderBy(F.col("Total_Visits").desc())

df_total_visits_flagged = (
    df_total_visits
    .withColumn("row_num", F.row_number().over(w))
    .withColumn(
        "Top_10pct_Visits_Badge",
        F.when(F.col("row_num") <= top_n, "Yes").otherwise("No")
    )
    .drop("row_num")
)


In [0]:
display(df_total_visits.filter(F.col("CDE_ID")== "6124897"))

#display(df_merge.filter(F.col("CDE_ID") == "6124897"))

In [0]:
# -----------------------------------------------------------
# Concession-only visits → count per member
# -----------------------------------------------------------
df_concession_visits_count = (
    df_concession_visits
    .groupBy("CDE_ID")
    .agg(F.count("*").alias("Concession_Visit_Count"))
)

# -----------------------------------------------------------
# Concession Ratio + 75% Badge
# -----------------------------------------------------------
df_visit_badges = (
    df_total_visits_flagged                 # you already computed this
    .join(df_concession_visits_count, "CDE_ID", "left")
    .fillna(0, subset=["Concession_Visit_Count"])
    .withColumn(
        "Concession_Ratio",
        F.col("Concession_Visit_Count") / F.col("Total_Visits")
    )
    .withColumn(
        "Concession_75pct_Badge",
        F.when(F.col("Concession_Ratio") >= 0.75, "Yes").otherwise("No")
    )
)


In [0]:
df_visit_badges.display()

### Popcorn & Drinks Purchase History
This is a CDE_ID Level Aggregation. Directly Left Join with the Main Table. **Only the Variable df_pop_drink**.

In [0]:
df_pop_drink = (
    df_pre_conc
    .groupBy("CDE_ID")
    .agg(
        F.sum(
            F.when(F.col("Item_Description").like("%POPCORN%LARGE%"), F.col("Quantity"))
             .when(F.col("Item_Description").like("%LARGE%POPCORN%"), F.col("Quantity"))
             .when(F.col("Item_Description").like("%VHO%CORN%VIP%POPCORN%LARGE%"), F.col("Quantity"))
             .otherwise(0)
        ).alias("Large_Popcorn"),

        F.sum(
            F.when(F.col("Item_Description").like("%POPCORN%REGULAR%"), F.col("Quantity"))
             .when(F.col("Item_Description").like("%REGULAR%POPCORN%"), F.col("Quantity"))
             .when(F.col("Item_Description").like("%VHO%CORN%VIP%POPCORN%REGULAR%"), F.col("Quantity"))
             .otherwise(0)
        ).alias("Regular_Popcorn"),

        F.sum(
            F.when(F.col("Item_Description").like("%POPCORN%SMALL%"), F.col("Quantity"))
             .when(F.col("Item_Description").like("%SMALL%POPCORN%"), F.col("Quantity"))
             .when(F.col("Item_Description").like("%VHO%CORN%VIP%POPCORN%SMALL%"), F.col("Quantity"))
             .otherwise(0)
        ).alias("Small_Popcorn"),

        F.sum(F.when(F.col("Item_Description") == "VHO CORN - UPSIZE TO REGULAR CORN", F.col("Quantity")).otherwise(0))
            .alias("Upsize_S_to_R_Popcorn"),

        F.sum(F.when(F.col("Item_Description") == "VHO CORN - UPSIZE TO LARGE CORN", F.col("Quantity")).otherwise(0))
            .alias("Upsize_R_to_L_Popcorn"),

        F.sum(F.when(F.col("Item_Description") == "VHO CORN - UPSIZE SMALL TO LARGE CORN", F.col("Quantity")).otherwise(0))
            .alias("Upsize_S_to_L_Popcorn"),

        F.sum(
            F.when(F.col("Item_Description").like("%VHO%DRINK%CUP%LARGE%"), F.col("Quantity"))
             .when(F.col("Item_Description").like("%VHO%DRINK%VIP%DRINK%LARGE%"), F.col("Quantity"))
             .otherwise(0)
        ).alias("Large_Fountain"),

        F.sum(
            F.when(F.col("Item_Description").like("%VHO%DRINK%CUP%REGULAR%"), F.col("Quantity"))
             .when(F.col("Item_Description").like("%VHO%DRINK%VIP%DRINK%REGULAR%"), F.col("Quantity"))
             .otherwise(0)
        ).alias("Regular_Fountain"),

        F.sum(
            F.when(F.col("Item_Description").like("%VHO%DRINK%CUP%SMALL%"), F.col("Quantity"))
             .otherwise(0)
        ).alias("Small_Fountain"),

        F.sum(
            F.when(F.col("Item_Description").like("%VHO%DRINK%CUP%KIDDIE%"), F.col("Quantity"))
             .otherwise(0)
        ).alias("Kiddie_Fountain"),

        F.sum(F.when(F.col("Item_Description").like("%UPSIZE SMALL FOUNTAIN%XL%"), F.col("Quantity")).otherwise(0))
            .alias("Upsize_S_XL"),

        F.sum(F.when(F.col("Item_Description").like("%UPSIZE REGULAR FOUNTAIN%XL%"), F.col("Quantity")).otherwise(0))
            .alias("Upsize_R_XL"),

        F.sum(F.when(F.col("Item_Description").like("%UPSIZE LARGE FOUNTAIN%XL%"), F.col("Quantity")).otherwise(0))
            .alias("Upsize_L_XL"),
    )
    .withColumn(
        "Total_Kernels",
        (
            (F.col("Small_Popcorn") * 300) +
            (F.col("Regular_Popcorn") * 400) +
            (F.col("Large_Popcorn") * 500) +
            (F.col("Upsize_S_to_R_Popcorn") * 400) +
            (F.col("Upsize_R_to_L_Popcorn") * 500) +
            (F.col("Upsize_S_to_L_Popcorn") * 500)
        ).cast("int")
    )
    .withColumn(
        "Total_Drink_ml",
        (
            (F.col("Small_Fountain") * 250) +
            (F.col("Regular_Fountain") * 350) +
            (F.col("Large_Fountain") * 500) +
            (F.col("Upsize_S_XL") * 500) +
            (F.col("Upsize_R_XL") * 500) +
            (F.col("Upsize_L_XL") * 500)
        ).cast("int")
    )
)


In [0]:
df_pop_drink.agg(F.sum("Total_Kernels")).show()
df_pop_drink.agg(F.sum("Total_Drink_ml")).show()

In [0]:
# ============================================================
# STEP 1 — Identify popcorn + butter + topping 
# ============================================================

df_popcorn_txn = (
    df_pre_conc
    .withColumn(
        "Is_Popcorn",
        F.when(
            F.col("Item_Description").rlike("(?i)POPCORN"),
            1
        ).otherwise(0)
    )
    .withColumn(
        "Has_Butter",
        F.when(
            F.col("Item_Description").rlike("(?i)BUTTER|LAYER"),
            1
        ).otherwise(0)
    )
    .withColumn(
        "Has_Topping",
        F.when(
            F.col("Item_Description").rlike("(?i)TOPPING|SEASONING|SHAKE\\s?IT"),
            1
        ).otherwise(0)
    )
)


# ============================================================
# STEP 2 — Aggregate per transaction (CDE_ID + date + location)
# ============================================================

df_popcorn_txn_agg = (
    df_popcorn_txn
    .groupBy("CDE_ID", "datefull", "LocationId")
    .agg(
        F.max("Is_Popcorn").alias("Popcorn_Flag"),
        F.max("Has_Butter").alias("Butter_Flag"),
        F.max("Has_Topping").alias("Topping_Flag")
    )
    .filter(F.col("Popcorn_Flag") == 1)     # keep only popcorn transactions
)

# ============================================================
# STEP 3 — Transaction-level final label
# ============================================================

df_popcorn_labeled = (
    df_popcorn_txn_agg
    .withColumn(
        "Popcorn_Label",
        F.when((F.col("Butter_Flag") == 1) & (F.col("Topping_Flag") == 1), "Butter & Topping")
         .when(F.col("Butter_Flag") == 1, "Butter")
         .when(F.col("Topping_Flag") == 1, "Topping")
         .otherwise("Plain")
    )
)

# ============================================================
# STEP 4 — Member-Level Preference (tie → most recent)
# ============================================================

window_recent = (
    Window.partitionBy("CDE_ID").orderBy(F.col("datefull").desc())
)

df_popcorn_preference = (
    df_popcorn_labeled
    .withColumn("rn", F.row_number().over(window_recent))
    .filter(F.col("rn") == 1)
    .select("CDE_ID", F.col("Popcorn_Label").alias("Popcorn_Preference"))
)


df_ticket_members = (
    df_all_trans
    .select("CDE_ID")
    .distinct()
)

# ============================================================
# STEP 5 — Add N/A for members who never bought popcorn
# ============================================================

df_popcorn_final = (
    df_ticket_members     # all CineClub members
    .join(df_popcorn_preference.alias("P"), "CDE_ID", "left")
    .withColumn(
        "Popcorn_Preference",
        F.coalesce(F.col("Popcorn_Preference"), F.lit("N/A"))
    )
)


In [0]:
K = F.col("Total_Kernels")
D = F.col("Total_Drink_ml")

df_favorite_snack = (
    df_pop_drink
    .select("CDE_ID", df_pop_drink.Total_Kernels, df_pop_drink.Total_Drink_ml)
    .withColumn(
        "Favorite_Snack",
        F.when(K > D, "Popcorn")
         .when(D> df_pop_drink.Total_Kernels, "Fountain Drink")
         .when((K == 0) & (D == 0), F.lit(None))
         .when(K == D, "Popcorn & Drink Tie")
         .otherwise("Unknown")
    )
)

Cumulative Total Kernels Bought By Cineclub Members in the current year.

In [0]:
Total_Popcorn_Kernels = df_pop_drink.agg(F.sum("Total_Kernels")).first()[0]
display(df_pop_drink.count())

### Movie Watch History
This a CDE_ID Level Aggregation. Directly Left Join with the Main Table. **Only the Variables: df_final_movie_summary and df_f1_watch_flag .**

In [0]:
from pyspark.sql import functions as F

# =====================================================================
# STEP 0 — GET ALL MOVIE WATCH EVENTS (TICKETS ONLY) + ADD RUNTIME
# =====================================================================
# Each ticket row = a movie watched on that date.
# We keep only 1 row per (CDE_ID, Title_ID, datefull).
# Then we join runtime for later calculations.

df_movie_views = (
    df_all_trans
    .join(T_films, df_all_trans.film_id == T_films.film_id, "left")
    .filter(F.col("TicketFLAG") == "Yes")
    .select("CDE_ID", "Title_ID", "Title_Name", df_all_trans.film_id, "datefull","Quantity")
    .groupBy("CDE_ID", "Title_ID", "Title_Name", df_all_trans.film_id, "datefull")
    .agg(
        F.sum(F.col("Quantity")).alias("No_of_Tickets")
    )
    .filter(F.col("No_of_Tickets")>0)
    .dropDuplicates()           # ensures 1 watch per movie per day
)

df_movie_views_with_runtime = (
    df_movie_views
    .join(
        T_films.select("film_id", "Film_Runtime"),
        "film_id",
        "left"
    )
    .filter(F.col("No_of_Tickets")>0)
    .withColumn("runtime", F.coalesce(F.col("Film_Runtime"), F.lit(0)))   # runtime=0 if missing
)

# =====================================================================
# STEP 1 — BUILD ORDERED WATCH-EVENT LIST FOR EACH USER
# =====================================================================
# For each user we want an ordered array of:
#   [ {watch_date, title, film_id}, ... ]
# Sorted by date.

df_watch_events = (
    df_movie_views_with_runtime
    .groupBy("CDE_ID")
    .agg(
        F.sort_array(
            F.collect_list(
                F.struct(
                    F.col("datefull").alias("watch_date"),
                    F.col("Title_Name").alias("title"),
                    F.col("film_id").alias("film")
                )
            )
        ).alias("sorted_watch_events")
    )
)

# =====================================================================
# STEP 2 — FIRST/LAST WATCH, REWATCH COUNT, TOTAL MINUTES PER MOVIE
# =====================================================================
# For each (CDE_ID, Title_ID):
# - first time watched
# - last time watched
# - title
# - how many days they watched this movie
# - total minutes watched (sum of runtimes)

df_first_watch = (
    df_movie_views_with_runtime
    .groupBy("CDE_ID", "Title_ID")
    .agg(
        F.min("datefull").alias("first_watch_date"),
        F.max("datefull").alias("last_watch_date"),
        F.first("Title_Name").alias("Title_Name"),
        F.first("film_id").alias("film_id"),
        F.count("*").alias("watch_count"),        # rewatch count (days)
        F.sum("runtime").alias("minutes_watched") # total minutes watched
    )
)

# =====================================================================
# STEP 3 — USER-LEVEL SUMMARY (DISTINCT MOVIES, TOTAL WATCHES, ETC.)
# =====================================================================
# For each user:
# - sorted list of movies (based on first watch date)
# - number of distinct movies
# - total watching events (rewatches included)
# - total minutes watched (sum over all movies)
# - stats array for extracting most watched movie

df_distinct_ordered_movies = (
    df_first_watch
    .groupBy("CDE_ID")
    .agg(
        # List of movies ordered by first watch date
        F.sort_array(
            F.collect_list(
                F.struct(
                    "first_watch_date",
                    "Title_Name",
                    "film_id"
                )
            )
        ).alias("sorted_movies"),

        # How many unique films they watched
        F.countDistinct("Title_ID").alias("Distinct_Movies_Watched"),

        # Total watches (including rewatches)
        F.sum("watch_count").alias("Total_Movies_Watched"),

        # Total minutes watched across all movies
        F.sum("minutes_watched").alias("Total_Minutes_Watched"),

        # Extra stats needed for "most watched"
        F.collect_list(
            F.struct(
                F.col("watch_count").alias("wc"),
                F.col("last_watch_date").alias("last_dt"),
                F.col("Title_Name").alias("title"),
                F.col("film_id").alias("film")
            )
        ).alias("movie_stats")
    )

    # Extract ordered list of titles
    .withColumn(
        "Movies_Watched_Title_Name",
        F.expr("transform(sorted_movies, x -> x.Title_Name)")
    )

    # Extract ordered list of film IDs
    .withColumn(
        "Movies_Watched_film_id",
        F.expr("transform(sorted_movies, x -> x.film_id)")
    )

    # ---------------------------------------------------------
    # MOST WATCHED MOVIE (tie-break: last watch date)
    # ---------------------------------------------------------
    .withColumn(
        "Most_Watched_Movie",
        F.expr("""
            aggregate(
                movie_stats,
                named_struct('wc', -1L, 'last_dt', date'1900-01-01', 'title', '', 'film', -1),
                (acc, x) ->
                    case
                        when x.wc > acc.wc then x
                        when x.wc = acc.wc and x.last_dt > acc.last_dt then x
                        else acc
                    end
            ).title
        """)
    )

    # Most watched count
    .withColumn(
        "Most_Watched_Count",
        F.expr("""
            aggregate(
                movie_stats,
                named_struct('wc', -1L, 'last_dt', date'1900-01-01', 'title', '', 'film', -1),
                (acc, x) ->
                    case
                        when x.wc > acc.wc then x
                        when x.wc = acc.wc and x.last_dt > acc.last_dt then x
                        else acc
                    end
            ).wc
        """)
    )
)

# =====================================================================
# STEP 4 — MERGE WATCH EVENTS AND PRODUCE FINAL SUMMARY
# =====================================================================
# Merge the watch-event list created in Step 1.
# Also format:
#   "Movie Name (2025-01-01)"
# for each watch event.

df_final_movie_summary = (
    df_distinct_ordered_movies
    .join(df_watch_events, "CDE_ID", "left")
    .withColumn(
        "Movies_Watched_With_Dates",
        F.expr("""
            transform(
                sorted_watch_events,
                x -> concat(x.title, ' (', cast(x.watch_date as string), ')')
            )
        """)
    )
    .select(
        "CDE_ID",
        "Total_Movies_Watched",
        "Total_Minutes_Watched",
        "Distinct_Movies_Watched",
        "Most_Watched_Movie",
        "Most_Watched_Count",
        "Movies_Watched_Title_Name",
        "Movies_Watched_film_id",
        "Movies_Watched_With_Dates"
    )
    .dropDuplicates()
)


In [0]:
display(df_final_movie_summary.filter(F.col("CDE_ID")== "11509008"))

In [0]:
display(df_final_movie_summary)

In [0]:
# ------------------------------------------------------------
# STEP 1 — Determine most recent watch date for each DISTINCT movie
# ------------------------------------------------------------
df_distinct_recent_movies = (
    df_movie_views
    .groupBy("CDE_ID", "Title_ID")   # Title_ID uniquely identifies the movie
    .agg(
        F.max("datefull").alias("most_recent_watch_date"),
        F.first("Title_Name").alias("Title_Name"),
        F.first("film_id").alias("film_id")
    )
)

# ------------------------------------------------------------
# STEP 2 — Rank distinct movies per user by most recent watch date
# ------------------------------------------------------------
w = Window.partitionBy("CDE_ID").orderBy(F.col("most_recent_watch_date").desc())

df_ranked_distinct = (
    df_distinct_recent_movies
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") <= 3)     # keep top 3 distinct films
)

# ------------------------------------------------------------
# STEP 3 — Pivot so each movie becomes its own column
# ------------------------------------------------------------
df_recent_3_distinct = (
    df_ranked_distinct
    .groupBy("CDE_ID")
    .pivot("rn", [1, 2, 3])
    .agg(F.first("Title_Name"))
    .select(
        "CDE_ID",
        F.col("1").alias("Recent_Movie_1"),
        F.col("2").alias("Recent_Movie_2"),
        F.col("3").alias("Recent_Movie_3")
    )
)


In [0]:
df_f1_watch_flag = (
    df_movie_views
    .groupBy("CDE_ID")
    .agg(
        F.sum(F.when(F.col("Title_ID") == 38635, 1).otherwise(0)).alias("F1_Movie_Watch_Count")
    )
    .withColumn(
        "F1_Watch_Flag",
        F.when(F.col("F1_Movie_Watch_Count") > 0, "Yes").otherwise("No")
    )
)


In [0]:
from pyspark.sql import functions as F

# ---------------------------------------------------------------
# 1. Prepare movie views (already distinct per day)
# ---------------------------------------------------------------
df_movie_views_short = df_movie_views.select(
    "CDE_ID", "film_id", "Title_ID", "datefull"
).dropDuplicates()

# ---------------------------------------------------------------
# 2. Release date lookup
# ---------------------------------------------------------------
df_release_dates = (
    T_films.select(
        "film_id",
        F.col("theatrical_release_date").alias("release_date")
    )
    .filter(F.col("release_date").isNotNull())
)

# ---------------------------------------------------------------
# 3. Join views with release date
# ---------------------------------------------------------------
df_watch_with_release = (
    df_movie_views_short
    .join(df_release_dates, "film_id", "left")
    .withColumn("days_from_release",
                F.datediff(F.col("datefull"), F.col("release_date")))
)

# ---------------------------------------------------------------
# 4. Flags for windows
# ---------------------------------------------------------------
df_watch_flags = (
    df_watch_with_release
    .withColumn(
        "Weekend_Release_Flag",
        F.when(
            (F.col("days_from_release") >= -5) &
            (F.col("days_from_release") <= 2),
            1
        ).otherwise(0)
    )
    .withColumn(
        "Week1_Release_Flag",
        F.when(
            (F.col("days_from_release") >= -5) &
            (F.col("days_from_release") <= 6),
            1
        ).otherwise(0)
    )
)

# ---------------------------------------------------------------
# 5. Compute summary metrics — DISTINCT MOVIE LEVEL
# ---------------------------------------------------------------
df_release_window_summary = (
    df_watch_flags
    .groupBy("CDE_ID")
    .agg(
        # Count distinct movies watched on release weekend
        F.countDistinct(
            F.when(F.col("Weekend_Release_Flag") == 1, F.col("Title_ID"))
        ).alias("Weekend_Release_Movies"),

        # Count distinct movies watched in first week
        F.countDistinct(
            F.when(F.col("Week1_Release_Flag") == 1, F.col("Title_ID"))
        ).alias("First_Week_Movies"),

        # Count distinct movies watched total
        F.countDistinct("Title_ID").alias("Distinct_Movies_Watched")
    )
)

# ---------------------------------------------------------------
# 6. Add the “True Opening-Weekend Fan” flag (≥ 50%)
# ---------------------------------------------------------------
df_release_window_summary = (
    df_release_window_summary
    .withColumn(
        "Opening_Weekend_Ratio",
        F.try_divide(F.col("First_Week_Movies"), F.col("Distinct_Movies_Watched"))
    )
    .withColumn(
        "Opening_Weekend_Fan_Flag",
        F.when(F.col("Opening_Weekend_Ratio") > 0.50, "Yes").otherwise("No")
    )
)

In [0]:
display(df_release_window_summary.filter(F.col("CDE_ID") == '11509008'))

### Favorites Flags
Aggregated on the CDE_ID Level. Direct Left Join to the Main Table. **Use Variable df_favourite_day_final, df_favourite_location_final,df_top3_genres.**

In [0]:
# ------------------------------------------------------------
# STEP 1 — Add day_of_week
# ------------------------------------------------------------
df_watch_days = (
    df_movie_views
    .withColumn("day_of_week", F.date_format("datefull", "EEEE"))
)

# ------------------------------------------------------------
# STEP 2 — Count watches per day per CDE_ID
# ------------------------------------------------------------
df_day_counts = (
    df_watch_days
    .groupBy("CDE_ID", "day_of_week")
    .agg(
        F.count("*").alias("watch_count"),
        F.max("datefull").alias("last_watch_date")
    )
)

# ------------------------------------------------------------
# STEP 3 — Pick favourite day using a window (simplest)
# ------------------------------------------------------------
w = Window.partitionBy("CDE_ID").orderBy(
    F.col("watch_count").desc(),
    F.col("last_watch_date").desc()
)

df_fav_day = (
    df_day_counts
    .withColumn("rn", F.row_number().over(w))
    .filter("rn = 1")        # keep only favourite day
    .drop("rn")
)

# ------------------------------------------------------------
# STEP 4 — Add total watches + percentage
# ------------------------------------------------------------
df_total = (
    df_movie_views
    .groupBy("CDE_ID")
    .agg(F.count("*").alias("Total_Watch_Events"))
)

df_favourite_day_final = (
    df_fav_day
    .join(df_total, "CDE_ID", "left")
    .withColumn(
        "Favourite_Day_Percent",
        F.round((F.col("watch_count") / F.col("Total_Watch_Events")) * 100, 2)
    )
    .select(
        "CDE_ID",
        F.col("day_of_week").alias("Favourite_Day"),
        F.col("watch_count").alias("Favourite_Day_Count"),
        "Favourite_Day_Percent"
    )
)


In [0]:
df_favourite_day_final.limit(100).display()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

# ================================================================
# STEP 1 — TRUE WATCH EVENTS (1 row per movie per day per location)
# ================================================================
df_watch_locations = (
    df_all_trans
    .filter(F.col("TicketFLAG") == "Yes")
    # .filter(F.col("CineClubFLAG") == "Yes")
    .select("CDE_ID", T_trans.LocationId, "film_id", "datefull")
    .dropDuplicates(["CDE_ID", "film_id", "datefull"])   # 1 real watching event
)

# ================================================================
# STEP 2 — WATCH COUNTS PER LOCATION PER USER
# ================================================================
df_location_counts = (
    df_watch_locations
    .groupBy("CDE_ID", T_trans.LocationId)
    .agg(
        F.count("*").alias("watch_count"),
        F.max("datefull").alias("last_watch_date")   # tie-breaker
    )
)

# ================================================================
# STEP 3 — PICK FAVOURITE LOCATION USING WINDOW
# ================================================================
w_loc = Window.partitionBy("CDE_ID") \
              .orderBy(F.col("watch_count").desc(),
                       F.col("last_watch_date").desc())

df_favourite_location = (
    df_location_counts
    .withColumn("rn", F.row_number().over(w_loc))
    .filter("rn = 1")
    .drop("rn")
)

# ================================================================
# STEP 4 — TOTAL WATCHES PER USER
# ================================================================
df_total_watch_events = (
    df_watch_locations
    .groupBy("CDE_ID")
    .agg(F.count("*").alias("Total_Watch_Events"))
)

# ================================================================
# STEP 5 — FINAL OUTPUT
# ================================================================
df_favourite_location_final = (
    df_favourite_location
    .join(df_total_watch_events, "CDE_ID", "left")
    .withColumn(
        "Favourite_Location_Percent",
        F.round((F.col("watch_count") / F.col("Total_Watch_Events")) * 100, 2)
    )
    .withColumnRenamed("LocationId", "Favourite_Location")
    .select(
        "CDE_ID",
        "Favourite_Location",
        "Favourite_Location_Percent"
    )
)


In [0]:
df_favourite_location_final.limit(100).display()

In [0]:
df_movie_genres = (
    df_movie_views
    .join(T_genre, "film_id", "left")
    .select(
        "CDE_ID",
        "film_id",
        "Title_ID",
        "Title_Name",
        "datefull",
        F.col("genre_hierarchy_1").alias("Genre")
    )
    .filter(F.col("Genre").isNotNull())     # ignore movies without genre
)

# ================================================================
# STEP 2 — Count genre frequency for each user
# ================================================================
df_genre_counts = (
    df_movie_genres
    .groupBy("CDE_ID", "Genre")
    .agg(
        F.countDistinct("datefull").alias("genre_watch_count"),   # distinct watch days
        F.max("datefull").alias("last_watch_date")                # tie-breaker
    )
)

# ================================================================
# STEP 3 — Rank genres by:
# 1. Most watch_count
# 2. Most recent watch date (tie-breaker)
# ================================================================
w = Window.partitionBy("CDE_ID") \
          .orderBy(
              F.col("genre_watch_count").desc(),
              F.col("last_watch_date").desc()
          )

df_ranked_genres = (
    df_genre_counts
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") <= 3)          # only top 3 genres
)

# ================================================================
# STEP 4 — Pivot into 3 genre columns
# ================================================================
df_top3_genres = (
    df_ranked_genres
    .groupBy("CDE_ID")
    .pivot("rn", [1, 2, 3])
    .agg(F.first("Genre"))
    .select(
        "CDE_ID",
        F.col("1").alias("Top_Genre_1"),
        F.col("2").alias("Top_Genre_2"),
        F.col("3").alias("Top_Genre_3")
    )
)

In [0]:
df_top3_genres.limit(100).display()

In [0]:
display(df_top3_genres.count())

### Ticket and Visit Format
Aggregated on the CDE_ID Level. Direct Left Join to the Main Table.

In [0]:
from pyspark.sql import functions as F

# ============================================================
# 1. Build visit-level base table (deduped movie views)
# ============================================================

df_visit_base = (
    df_all_trans.filter(F.col("TicketFLAG") == "Yes")
    .select("CDE_ID", "film_id", "datefull", "Quantity")
    .groupBy("CDE_ID", "film_id", "datefull")
    .agg(
        F.sum(F.col("Quantity")).alias("No_of_Tickets")
    )
    .filter(F.col("No_of_Tickets")>0)
    .drop("No_of_Tickets")
    .dropDuplicates(["CDE_ID", "film_id", "datefull"])
)


# ============================================================
# 2. Build visit-level metadata collapsed from df_all_trans
# ============================================================

df_visit_meta_clean = (
    df_all_trans
    .join(T_sessions, "sessionskey" ,"left")
    .filter(F.col("TicketFLAG") == "Yes")
    .groupBy("CDE_ID", "film_id", "datefull")
    .agg(
        F.max("Auditorium_Experience_ID").alias("Meta_Auditorium_Experience_ID"),
        F.max("Performance_Experience_ID").alias("Meta_Performance_Experience_ID"),
        F.max("IS_DBOX_Flag").alias("Meta_IS_DBOX_Flag")
    )
)

# ============================================================
# 3. Combine visit base + metadata into one visit table
# ============================================================

df_visit_full = (
    df_visit_base.alias("VB")
    .join(
        df_visit_meta_clean.alias("VM"),
        ["CDE_ID", "film_id", "datefull"],
        "left"
    )
)

# ============================================================
# 4. Define Cineplex format classification hierarchy
#    Priority:
#       1. Top Tier → AVX, IMAX, VIP, 4DX, ScreenX
#       2. DBOX
#       3. 3D
#       4. Regular
# ============================================================

top_tier_mapping = {
    2: "AVX",
    3: "IMAX",
    4: "VIP",
    7: "FourDX",
    8: "ScreenX"
}

mapping_expr = F.create_map([F.lit(x) for x in sum(top_tier_mapping.items(), ())])

df_visit_formats = (
    df_visit_full
    # Identify top-tier format label
    .withColumn("TopTier_Label",
                mapping_expr.getItem(F.col("Meta_Auditorium_Experience_ID"))
    )
    # Final hierarchical format assignment
    .withColumn(
        "Final_Format",
        F.when(F.col("TopTier_Label").isNotNull(), F.col("TopTier_Label"))
         .when(F.col("Meta_IS_DBOX_Flag") == "Yes", F.lit("DBOX"))
         .when(F.col("Meta_Performance_Experience_ID") == 2, F.lit("ThreeD"))
         .otherwise(F.lit("Regular"))
    )
)

# ============================================================
# 5. Add one-hot flags for each format
# ============================================================

df_visit_flags = (
    df_visit_formats
    .withColumn("Flag_Regular", F.when(F.col("Final_Format") == "Regular", 1).otherwise(0))
    .withColumn("Flag_ThreeD",  F.when(F.col("Final_Format") == "ThreeD", 1).otherwise(0))
    .withColumn("Flag_DBOX",    F.when(F.col("Final_Format") == "DBOX", 1).otherwise(0))
    .withColumn("Flag_AVX",     F.when(F.col("Final_Format") == "AVX", 1).otherwise(0))
    .withColumn("Flag_IMAX",    F.when(F.col("Final_Format") == "IMAX", 1).otherwise(0))
    .withColumn("Flag_VIP",     F.when(F.col("Final_Format") == "VIP", 1).otherwise(0))
    .withColumn("Flag_FourDX",  F.when(F.col("Final_Format") == "FourDX", 1).otherwise(0))
    .withColumn("Flag_ScreenX", F.when(F.col("Final_Format") == "ScreenX", 1).otherwise(0))
)

# ============================================================
# 6. Aggregate to user level (format counts)
# ============================================================

df_format_summary = (
    df_visit_flags
    .groupBy("CDE_ID")
    .agg(
        F.count("*").alias("Ticket_Visit_Count"),
        F.sum("Flag_Regular").alias("Regular_Visits"),
        F.sum("Flag_ThreeD").alias("ThreeD_Visits"),
        F.sum("Flag_DBOX").alias("DBOX_Visits"),
        F.sum("Flag_AVX").alias("AVX_Visits"),
        F.sum("Flag_IMAX").alias("IMAX_Visits"),
        F.sum("Flag_VIP").alias("VIP_Visits"),
        F.sum("Flag_FourDX").alias("FourDX_Visits"),
        F.sum("Flag_ScreenX").alias("ScreenX_Visits")
    )
)

# ============================================================
# 7. Percentages (each visit = 1)
# ============================================================

df_format_summary = (
    df_format_summary
    .withColumn("Pct_Regular",    F.round(F.col("Regular_Visits")    / F.col("Ticket_Visit_Count") * 100, 2))
    .withColumn("Pct_ThreeD",     F.round(F.col("ThreeD_Visits")     / F.col("Ticket_Visit_Count") * 100, 2))
    .withColumn("Pct_DBOX",       F.round(F.col("DBOX_Visits")       / F.col("Ticket_Visit_Count") * 100, 2))
    .withColumn("Pct_AVX",        F.round(F.col("AVX_Visits")        / F.col("Ticket_Visit_Count") * 100, 2))
    .withColumn("Pct_IMAX",       F.round(F.col("IMAX_Visits")       / F.col("Ticket_Visit_Count") * 100, 2))
    .withColumn("Pct_VIP",        F.round(F.col("VIP_Visits")        / F.col("Ticket_Visit_Count") * 100, 2))
    .withColumn("Pct_FourDX",     F.round(F.col("FourDX_Visits")     / F.col("Ticket_Visit_Count") * 100, 2))
    .withColumn("Pct_ScreenX",    F.round(F.col("ScreenX_Visits")    / F.col("Ticket_Visit_Count") * 100, 2))
)

# ============================================================
# 8. Diversity Badge (4+ formats used)
# ============================================================

df_format_summary = (
    df_format_summary
    .withColumn(
        "Formats_Used_Count",
        (
            (F.col("ThreeD_Visits")  > 0).cast("int") +
            (F.col("DBOX_Visits")    > 0).cast("int") +
            (F.col("AVX_Visits")     > 0).cast("int") +
            (F.col("IMAX_Visits")    > 0).cast("int") +
            (F.col("VIP_Visits")     > 0).cast("int") +
            (F.col("FourDX_Visits")  > 0).cast("int") +
            (F.col("ScreenX_Visits") > 0).cast("int")
        )
    )
    .withColumn(
        "Formats_4plus_Flag",
        F.when(F.col("Formats_Used_Count") >= 4, "Yes").otherwise("No")
    )
)


In [0]:
df_format_summary.filter(F.col("CDE_ID") == "6124897").display()

### Popular Movies
To be forwarded separately.

In [0]:
# =====================================================================
# 1. UNIQUE MEMBER → MOVIE pairs
# =====================================================================
df_population_cineclub = df_cineclub_members.select("CDE_ID").distinct()

df_member_movie = (
    df_movie_views
    .join(df_population_cineclub, "CDE_ID", "inner")
    .select("CDE_ID", "title_id", "Title_Name")
    .dropDuplicates(["CDE_ID", "title_id"])   # each member counted once per movie
)

# =====================================================================
# 2. COUNT DISTINCT MEMBERS PER MOVIE
# =====================================================================
df_movie_member_counts = (
    df_member_movie
    .groupBy("title_id", "Title_Name")
    .agg(
        F.countDistinct("CDE_ID").alias("Members_Watched")   # <-- THE KEY METRIC
    )
    .orderBy(F.col("Members_Watched").desc())
    .limit(10)
)

# =====================================================================
# 3. RANK THE TOP 10 MOVIES
# =====================================================================
w = Window.orderBy(F.col("Members_Watched").desc())

df_top10_movies_member_based = (
    df_movie_member_counts
    .withColumn("Movie_No", F.row_number().over(w))
    .select(
        "Movie_No",
        F.col("Title_Name").alias("Movie_Name"),
        "title_id",
        "Members_Watched"
    )
    .withColumn("Viewer_prct", F.round(F.col("Members_Watched") / df_cineclub_members.count() * 100, 2))
)

df_top10_movies_member_based


In [0]:
df_top10_movies_member_based.display()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# =====================================================
# 0. ALL Cineplex Movie View Events (Full Population, Refund-Safe)
# =====================================================

df_movie_views_raw = (
    T_trans
    .join(T_films, T_trans.film_id == T_films.film_id)
    .join(T_date, T_trans.VisitDateId == T_date.dateid)
    .filter(
        (F.col("TicketFLAG") == "Yes") &
        (F.col("datefull").between("2025-01-01", yesterday))
    )
    .select(
        "CDE_ID",
        T_trans.film_id.alias("film_id"),
        "Title_Name",
        "Title_ID",
        "LocationId",
        "datefull",
        "Quantity"
    )
)

# Refund-safe net quantity (Quantity may be positive or -1 for refund)
df_movie_views_all = (
    df_movie_views_raw
    .groupBy("CDE_ID", "film_id", "datefull", "Title_Name", "Title_ID", "LocationId")
    .agg(F.sum("Quantity").alias("No_of_Tickets"))
    .filter(F.col("No_of_Tickets") > 0)     # Only valid visits (refund-safe)
)

# =====================================================
# 1. Total Tickets Sold (DENOMINATOR for %)
# =====================================================
total_tickets_sold = (
    df_movie_views_all
    .agg(F.sum("No_of_Tickets").alias("Total_Tickets"))
    .collect()[0]["Total_Tickets"]
)

# =====================================================
# 2. Top 10 Movies by Ticket Sales
# =====================================================
df_top10_movies_all = (
    df_movie_views_all
    .groupBy("Title_ID", "Title_Name")
    .agg(
        F.sum("No_of_Tickets").alias("Total_Tickets_Sold")   # KEY METRIC
    )
    .orderBy(F.col("Total_Tickets_Sold").desc())
    .limit(10)
)

# =====================================================
# 3. Rank + Ticket Share %
# =====================================================
w = Window.orderBy(F.col("Total_Tickets_Sold").desc())

df_top10_final_all = (
    df_top10_movies_all
    .withColumn("Movie_No", F.row_number().over(w))
    .withColumn(
        "Tickets_Share_Percent",
        F.round((F.col("Total_Tickets_Sold") / total_tickets_sold) * 100, 2)
    )
    .select(
        "Movie_No",
        F.col("Title_Name").alias("Movie_Name"),
        "Title_ID",
        "Total_Tickets_Sold",
        "Tickets_Share_Percent"
    )
)

df_top10_final_all


In [0]:
display(df_top10_final_all)

### CineClub and Blockbuster Flags
This is Cineclub Member Specific Flag, join to Cineclub Member Table with CDE_ID. No aggregation Needed.

In [0]:
from pyspark.sql import functions as F

# -------------------------------------------------------
# 0. Base list of all CineClub members
# -------------------------------------------------------
df_members = df_cineclub_members.select("CDE_ID").distinct()

# -------------------------------------------------------
# 1. Top 10 movies WITH film_id and Movie_No
# -------------------------------------------------------
df_top10_with_ids = df_top10_movies_member_based.select("Movie_No", "title_id", "Movie_Name")

# -------------------------------------------------------
# 2. All user-film combinations (actual user watch events)
# -------------------------------------------------------
df_user_films = df_movie_views.select("CDE_ID", "title_id").distinct()

# -------------------------------------------------------
# 3. CROSS JOIN → every CDE_ID gets 10 movies
# -------------------------------------------------------
df_member_movie_matrix = df_members.crossJoin(df_top10_with_ids)

# -------------------------------------------------------
# 4. LEFT JOIN to detect whether user watched film_id
# -------------------------------------------------------
df_user_top10 = (
    df_member_movie_matrix.alias("base")
    .join(
        df_user_films.alias("uf"),
        on=[F.col("base.CDE_ID") == F.col("uf.CDE_ID"),
            F.col("base.title_id") == F.col("uf.title_id")],
        how="left"
    )
    .withColumn(
        "Flag",
        F.when(F.col("uf.title_id").isNotNull(), "Yes").otherwise("No")
    )
    .select("base.CDE_ID", "Movie_No", "Flag")
)

# -------------------------------------------------------
# 5. Pivot → 10 flag columns
# -------------------------------------------------------
df_user_top10_flags = (
    df_user_top10
    .groupBy("CDE_ID")
    .pivot("Movie_No")
    .agg(F.first("Flag"))
    .fillna("No")
)

# -------------------------------------------------------
# 6. Rename pivot columns
# -------------------------------------------------------
for i in range(1, 11):
    df_user_top10_flags = df_user_top10_flags.withColumnRenamed(str(i), f"Movie_{i}_CC_Flag")

# -------------------------------------------------------
# 7. Total_CC_Watch_Count
# -------------------------------------------------------
yes_exprs = [F.when(F.col(f"Movie_{i}_CC_Flag") == "Yes", 1).otherwise(0) for i in range(1, 11)]

df_user_top10_flags = df_user_top10_flags.withColumn("Total_CC_Watch_Count", sum(yes_exprs))


In [0]:
df_user_top10_flags.display()

In [0]:
display(df_user_top10_flags.count())

In [0]:
from pyspark.sql import functions as F

# -------------------------------------------------------
# A. Base list of all CineClub members
# -------------------------------------------------------
df_members = df_cineclub_members.select("CDE_ID").distinct()

# -------------------------------------------------------
# B. Top 10 Blockbuster Movies (from full population)
# -------------------------------------------------------
df_top10_blockbusters = (
    df_top10_final_all
    .select("Movie_No", "Title_ID", "Movie_Name")
)

# -------------------------------------------------------
# C. All CineClub member movie views (Title_ID level)
# -------------------------------------------------------
df_CC_movie_views = (
    df_movie_views_all
    .join(df_members, "CDE_ID", "inner")      # ★ ensure only CineClub members
    .select("CDE_ID", "Title_ID")
    .distinct()
)

# -------------------------------------------------------
# D. CROSS JOIN: every CineClub member × 10 blockbuster movies
# -------------------------------------------------------
df_CC_matrix = df_members.crossJoin(df_top10_blockbusters)

# -------------------------------------------------------
# E. LEFT JOIN to detect whether user watched each blockbuster
# -------------------------------------------------------
df_CC_top10 = (
    df_CC_matrix.alias("base")
    .join(
        df_CC_movie_views.alias("mv"),
        on=[
            F.col("base.CDE_ID") == F.col("mv.CDE_ID"),
            F.col("base.Title_ID") == F.col("mv.Title_ID")
        ],
        how="left"
    )
    .withColumn(
        "Flag",
        F.when(F.col("mv.Title_ID").isNotNull(), "Yes").otherwise("No")
    )
    .select("base.CDE_ID", "Movie_No", "Flag")
)

# -------------------------------------------------------
# F. Pivot → Movie_1_BB_Flag ... Movie_10_BB_Flag
# -------------------------------------------------------
df_user_top10_flags_CC = (
    df_CC_top10
    .groupBy("CDE_ID")
    .pivot("Movie_No")
    .agg(F.first("Flag"))
    .fillna("No")
)

# Rename pivot columns
for i in range(1, 11):
    df_user_top10_flags_CC = (
        df_user_top10_flags_CC
        .withColumnRenamed(str(i), f"Movie_{i}_BB_Flag")
    )

# -------------------------------------------------------
# G. Add Total Blockbuster Watch Count
# -------------------------------------------------------
yes_exprs = [
    F.when(F.col(f"Movie_{i}_BB_Flag") == "Yes", 1).otherwise(0)
    for i in range(1, 11)
]

df_user_top10_flags_CC = df_user_top10_flags_CC.withColumn(
    "Total_BB_Watch_Count",
    sum(yes_exprs)
)

# -------------------------------------------------------
# H. Blockbuster Badge (≥5 movies watched)
# -------------------------------------------------------
df_user_top10_flags_CC = df_user_top10_flags_CC.withColumn(
    "Blockbuster_Badge",
    F.when(F.col("Total_BB_Watch_Count") >= 5, "Yes").otherwise("No")
)


In [0]:
display(df_user_top10_flags_CC.filter(F.col("BlockBuster_Badge") == 'No'))

In [0]:
display(df_user_top10_flags_CC.count())

### Concessions and Savings

In [0]:
# pip install openpyxl #Only Run Once in each Session

In [0]:
import pandas as pd
df_pricing_pd = pd.read_excel("/Workspace/Users/ranjan.mishra@cineplex.com/Offline Data/Circuit Pricing.xlsx")
df_pricing = spark.createDataFrame(df_pricing_pd)

from pyspark.sql import functions as F
df_pricing = df_pricing.withColumn(
    "Theatre_ID_Array",
    F.split(F.col("Theatre_ID"), ",")
)
df_price_exploded = (
    df_pricing
    .withColumn("Theatre_ID_exploded", F.explode("Theatre_ID_Array"))
    .withColumn("Theatre_ID_clean", F.col("Theatre_ID_exploded").cast("int"))
).drop("Category_Group", "Theatre_ID", "Theatre_ID_Array", "Theatre_ID_exploded", "Child","Senior","Cineplex_Tuesday", "Tuesday" )

In [0]:
df_trans_savings = (
    df_all_trans
    .where(
        (F.col("TicketFLAG") == "Yes") &
        (F.col("CineClub_Discount_Type_ID").isin(5, 6))
    )
    .groupBy("CDE_ID", T_trans.LocationId)
    .agg(F.sum("Quantity").alias("No_of_tickets"))
    .filter(F.col("No_of_tickets") > 0)
)

df_ticket_savings = (
    df_trans_savings.alias("T")
    .join(df_price_exploded.alias("P"), F.col("T.LocationId") == F.col("P.Theatre_ID_clean"), "left")
    .select(
        "T.CDE_ID",
        "T.LocationId",
        "T.No_of_tickets",
        "P.General"
    )
    .withColumn(
        "Dollar_Saved",
        F.round(((F.col("General") - F.lit(9.99)) * F.col("No_of_tickets"))+ (F.col("No_of_tickets") * F.lit(1.5)) , 2)
    )
)

df_total_ticket_savings = (
    df_ticket_savings
    .groupBy("CDE_ID")
    .agg(F.round(F.sum("Dollar_Saved"), 2).alias("Total_Ticket_Savings"))
)

In [0]:
df_concession_savings = (
    df_pre2
    .where(
        (F.col("Net_Revenue") > 0) &
        (F.col("CineClub_Discount_Type_ID").isin(2, 3))
    )
    .groupBy("CDE_ID")
    .agg(
        F.round(F.sum(F.when(F.col("CineClub_Discount_Type_ID") == 2, F.col("Net_Revenue"))),2).alias("Total_Concession_Revenue"),
        F.round(F.sum(F.when(F.col("CineClub_Discount_Type_ID") == 3, F.col("Net_Revenue"))),2).alias("Total_Concession_LBE_Revenue")
    )
    .withColumn("Total_Savings_Concessions", F.coalesce(F.round(F.col("Total_Concession_Revenue") * 0.25,2)))                      # 20% EXH discount
    .withColumn(
    "Total_Savings_LBE",
    F.coalesce(
        F.round(
            F.col("Total_Concession_LBE_Revenue") * 0.25, 
            2
        ),
        F.lit(0.0)
    )
)
                 # 10% discount (reverse-engineered)
)

In [0]:
display(df_concession_savings.agg(F.sum("Total_Savings_LBE")))

### Member Joining Month-Year and Joined This year Flag
Aggregated to CDE_ID Level, we basically just filter the event table and join the matching cc_profile_id to the customer table. 

In [0]:
# ---------------------------------------------------------
# 1. JOIN subscription table with customer table to get CDE_ID
# ---------------------------------------------------------

df_subscription_event = (
    T_events.alias("S")
    .join(
        df_cineclub_members.alias("C"), "cc_user_profile_id", "inner")
    .select(
        F.col("C.CDE_ID").alias("CDE_ID"),
        F.col("S.event_type"),
        F.col("S.reporting_datetime")
    )
)

# ---------------------------------------------------------
# 2. FILTER subscription_created events
# ---------------------------------------------------------

df_sub_created = (
    df_subscription_event
    .filter(F.col("event_type") == "subscription_created")
)

# df_sub_renewed = (df_subscription_event      #People who are current member within this year 
#     .filter(F.col("event_type") == "subscription_renewed")
#     .filter(F.col("reporting_datetime") > "2025-01-01") 
#     )
    
df_first_join_date = (
    df_sub_created
    .groupBy("CDE_ID")
    .agg(
        F.min("reporting_datetime").alias("CineClub_Join_Date")
    )
)

df_first_join_final = (
    df_first_join_date
    .withColumn(
        "Joined_2025_Flag",
        F.when(F.col("CineClub_Join_Date") >= F.to_timestamp(F.lit("2025-01-01")), "Yes")
         .otherwise("No")
    )
    .withColumn(
        "Join_Month_Year",
        F.date_format(F.col("CineClub_Join_Date"), "MM-yyyy")
    )
)


### Individual Statistics Final Merge

In [0]:
df_merge = (
  df_cineclub_members.select("CDE_ID", "plan_id", "recognitions_left")      #CHECKED
  .join(df_first_join_final, "CDE_ID", "left")                              #CHECKED
  .join(df_scene_points, "CDE_ID", "left")                                  #CHECKED
  .join(df_visit_badges, "CDE_ID", "left")                                  #CHECKED
  .join(df_format_summary, "CDE_ID", "left")                                #CHECKED
  .join(df_pop_drink, "CDE_ID", "left") #IF YOU RUN FULL NOTEBOOK, RESET THIS BEFORE THE SECTION
  .join(df_favorite_snack, "CDE_ID", "left")                                #CHECKED
  .join(df_popcorn_final, "CDE_ID", "left")                                 #CHECKED
  .join(df_final_movie_summary, "CDE_ID", "left")                           #CHECKED
  .join(df_release_window_summary, "CDE_ID", "left")                        #CHECKED
  .join(df_recent_3_distinct, "CDE_ID", "left")                             #CHECKED
  .join(df_top3_genres, "CDE_ID", "left")                                   #CHECKED
  .join(df_f1_watch_flag, "CDE_ID", "left")                                 #CHECKED
  .join(df_favourite_day_final, "CDE_ID", "left")                           #CHECKED
  .join(df_favourite_location_final, "CDE_ID", "left")                      #CHECKED
  .join(df_user_top10_flags, "CDE_ID", "left")                              #CHECKED
  .join(df_user_top10_flags_CC, "CDE_ID", "left")                           #CHECKED
  .join(df_total_ticket_savings, "CDE_ID", "left")                          #CHECKED                         
  .join(df_concession_savings, "CDE_ID", "left")
  .withColumn("Total_Savings", 
    F.round(F.col("Total_Savings_Concessions") + 
            F.col("Total_Savings_LBE") + 
            F.col("Total_Ticket_Savings"),2))                       
  .filter(F.col("Total_Visits").isNotNull())
)

In [0]:
df_merge.display()

In [0]:
display(df_merge.count())

### Average Member Statistics 
We only filter the subscription_event table for the CDE_IDs which satisfy the constraint and then join relevant columns from the Queries above. We then Do table aggregations to get the answers.


In [0]:
# 1. Base events dataframe (your starting point)
df_subscription_event = (
    T_events.alias("S")
    .join(
        df_cineclub_members.alias("C"), "cc_user_profile_id", "inner"
    )
    .select(
        F.col("C.CDE_ID").alias("CDE_ID"),
        F.col("S.event_type"),
        F.col("S.reporting_datetime")
    )
)

# 2. Filter to events on or after 2025-01-01 AND only subscription_renewed
df_renewals_2025 = (
    df_subscription_event
    .filter(F.col("reporting_datetime") >= F.to_timestamp(F.lit("2025-01-01")))
    .filter(F.col("event_type") == "subscription_renewed")
)

# 3. Count renewals per CDE_ID
df_renewal_counts = (
    df_renewals_2025
    .groupBy("CDE_ID")
    .agg(F.count("*").alias("renewals_after_2025"))
)

# 4. Only pick those with >= 10 renewals
df_10plus_renewals = df_renewal_counts.filter(F.col("renewals_after_2025") >= 10)

In [0]:
df_average_savings_metrics = (
    df_10plus_renewals
    .join(df_total_ticket_savings, "CDE_ID", "left")                          #NEED TO VERIFY
    .join(df_concession_savings, "CDE_ID", "left")
    .withColumn("Total_Savings", F.round(F.col("Total_Ticket_Savings") + F.col("Total_Savings_Concessions") + F.col("Total_Savings_LBE"),2))                           #NEED TO VERIFY
)

In [0]:
df_average_savings_metrics.display()

### Cumulative Data (By Province)

In [0]:
# ---------------------------------------------------------
# Base: All CineClub ticket rows in 2025
# ---------------------------------------------------------
df_cc_tickets = (
    df_all_trans
    .join(T_locations.withColumnRenamed("Province_CD", "Prov"), "df_all_trans.LocationId", "left")
    .join(T_films, "film_id", "left")
    .filter(F.col("CineClubFLAG") == "Yes")
    .filter(F.col("TicketFLAG") == "Yes")
    .filter(F.col("datefull").between("2025-01-01", today))
)

# ---------------------------------------------------------
# 1. Ticket rows per movie per province (numerator)
# ---------------------------------------------------------
df_movie_province = (
    df_cc_tickets
    .groupBy("Prov", "title_id", "Title_Name")
    .agg(F.count("*").alias("Ticket_Rows_Province"))
)

# ---------------------------------------------------------
# 2. Total CineClub ticket rows per movie (denominator)
# ---------------------------------------------------------
df_movie_totals = (
    df_cc_tickets
    .groupBy("title_id")
    .agg(F.count("*").alias("Total_Rows_Movie"))
)

# ---------------------------------------------------------
# 3. Join numerator + denominator and calculate percentage
# ---------------------------------------------------------
df_movie_with_pct = (
    df_movie_province
    .join(df_movie_totals, "title_id", "left")
    .withColumn(
        "Percent_of_Movie",
        F.round(
            (F.col("Ticket_Rows_Province") / F.col("Total_Rows_Movie")) * 100,
            2
        )
    )
)

# ---------------------------------------------------------
# 4. Rank top 3 movies per province
# ---------------------------------------------------------
w = Window.partitionBy("Prov").orderBy(F.col("Ticket_Rows_Province").desc())

df_top3_movies_per_province = (
    df_movie_with_pct
    .withColumn("Movie_No", F.row_number().over(w))
    .filter(F.col("Movie_No") <= 5)
    .select(
        "Prov",
        "Movie_No",
        "title_id",
        "Title_Name",
        "Ticket_Rows_Province",
        "Total_Rows_Movie",
        "Percent_of_Movie"
    )
    .orderBy("Prov", "Movie_No")
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

# =====================================================================
# TOP 3 MOVIES PER PROVINCE — UNIQUE VISITS + TOTAL MINUTES
# =====================================================================

# ---------------------------------------------------------
# STEP A — Rebuild movie views WITH LocationId included
# ---------------------------------------------------------
df_movie_views = (
    df_all_trans
    .join(T_films, df_all_trans.film_id == T_films.film_id, "left")
    .filter((F.col("TicketFLAG") == "Yes") & (F.col("CineClubFLAG") == "Yes"))
    .select(
        "CDE_ID",
        "Title_ID",
        "Title_Name",
        df_all_trans.film_id,
        "datefull",
        "Quantity",
        T_trans.LocationId                  # <-- REQUIRED
    )
    .groupBy(
        "CDE_ID",
        "Title_ID",
        "Title_Name",
        df_all_trans.film_id,
        "datefull",
        T_trans.LocationId                 # <-- KEEP IT HERE
    )
    .agg(
        F.sum("Quantity").alias("No_of_Tickets")
    )
    .dropDuplicates()
)

# ---------------------------------------------------------
# STEP B — Add runtime safely
# ---------------------------------------------------------
df_movie_views_with_runtime = (
    df_movie_views
    .join(
        T_films.select("film_id", "Film_Runtime"),
        "film_id",
        "left"
    )
    .filter(F.col("No_of_Tickets") > 0)
    .withColumn("runtime", F.coalesce(F.col("Film_Runtime"), F.lit(0)))
)

# ---------------------------------------------------------
# STEP C — Attach Province from T_locations
# ---------------------------------------------------------
df_movie_with_province = (
    df_movie_views_with_runtime
    .join(
        T_locations.select(
            "LocationId",
            "Province_CD"
        ),
        "LocationId",
        "left"
    )
    .filter(F.col("Province_CD").isNotNull())
)

# ---------------------------------------------------------
# STEP D — UNIQUE VISITS
# One visit = one CDE_ID watching one film on one date
# ---------------------------------------------------------
df_unique_visits = (
    df_movie_with_province
    .select(
        "Province_CD",
        "CDE_ID",
        "film_id",
        "Title_Name",
        "datefull",
        "runtime"
    )
    .dropDuplicates(["Province_CD", "CDE_ID", "film_id", "datefull"])
)

# ---------------------------------------------------------
# STEP E — Count Unique Visits + Total Minutes Watched
# ---------------------------------------------------------
df_movie_counts_unique = (
    df_unique_visits
    .groupBy("Province_CD", "film_id", "Title_Name")
    .agg(
        F.count("*").alias("Unique_Visits"),
        F.sum("runtime").alias("Total_Minutes_Watched_Province")
    )
)

# ---------------------------------------------------------
# STEP F — Rank Movies Inside Each Province
# ---------------------------------------------------------
window_rank = Window.partitionBy("Province_CD") \
                    .orderBy(F.col("Unique_Visits").desc())

df_ranked_movies = (
    df_movie_counts_unique
    .withColumn("Rank_in_Province", F.rank().over(window_rank))
)

# ---------------------------------------------------------
# STEP G — Final Top 3 Movies per Province
# ---------------------------------------------------------
df_top3_movies_by_province = (
    df_ranked_movies
    .filter(F.col("Rank_in_Province") <= 3)
    .orderBy("Province_CD", "Rank_in_Province")
)

df_top3_movies_by_province

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# =====================================================================
# STEP A — Build movie views with LocationId (refund-safe)
# =====================================================================
df_movie_views = (
    df_all_trans
    .join(T_films, "film_id", "left")
    .filter((F.col("TicketFLAG") == "Yes") & (F.col("CineClubFLAG") == "Yes"))
    .select(
        "CDE_ID",
        "Title_ID",
        "Title_Name",
        "film_id",
        "datefull",
        "Quantity",
        T_trans.LocationId
    )
    .groupBy(
        "CDE_ID", "Title_ID", "Title_Name", "film_id", "datefull", T_trans.LocationId
    )
    .agg(F.sum("Quantity").alias("No_of_Tickets"))
    .filter(F.col("No_of_Tickets") > 0)   # refund-safe
)

# =====================================================================
# STEP B — Attach runtime
# =====================================================================
df_movie_with_runtime = (
    df_movie_views
    .join(T_films.select("film_id", "Film_Runtime"), "film_id", "left")
    .withColumn("runtime", F.coalesce(F.col("Film_Runtime"), F.lit(0)))
)

# =====================================================================
# STEP C — Attach Province
# =====================================================================
df_movie_with_province = (
    df_movie_with_runtime
    .join(T_locations.select("LocationId", "Province_CD"), "LocationId", "left")
    .filter(F.col("Province_CD").isNotNull())
)

# =====================================================================
# STEP D — DISTINCT WATCHES (Visit-based)
# one CDE_ID watching one movie on one date
# =====================================================================
df_distinct_watches = (
    df_movie_with_province
    .join(T_genre, "film_id", "left")
    .select(
        "Province_CD",
        "CDE_ID",
        "film_id",
        "Title_ID",
        "Title_Name",
        "runtime",
        "datefull",
        "genre_hierarchy_1"
    )
    .dropDuplicates(["Province_CD", "CDE_ID", "title_id", "datefull"])
)

# =====================================================================
# STEP E — COUNT DISTINCT WATCHES and TOTAL RUNTIME
# =====================================================================
df_movie_watch_stats = (
    df_distinct_watches
    .groupBy("Province_CD", "Title_ID", "Title_Name")
    .agg(
        F.count("*").alias("Distinct_Watches"),
        F.sum("runtime").alias("Total_Runtime")
    )
)

# =====================================================================
# STEP F — RANK MOVIES INSIDE EACH PROVINCE BY DISTINCT WATCHES
# =====================================================================
window_rank = (
    Window.partitionBy("Province_CD")
    .orderBy(F.col("Distinct_Watches").desc())
)

df_ranked_movies = (
    df_movie_watch_stats
    .withColumn("Rank_in_Province", F.rank().over(window_rank))
)

# =====================================================================
# STEP G — FINAL TABLE (Top movies per province)
# =====================================================================
df_top_movies_by_province = (
    df_ranked_movies
    .orderBy("Province_CD", "Rank_in_Province")
    .filter(F.col("Rank_in_Province") <= 5)
    .drop("film_id","Total_Runtime")
)

df_top_movies_by_province


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# =====================================================================
# STEP A–D (YOUR ORIGINAL LOGIC) — DO NOT CHANGE
# =====================================================================

df_movie_views = (
    df_all_trans
    .join(T_films, "film_id", "left")
    .filter((F.col("TicketFLAG") == "Yes") & (F.col("CineClubFLAG") == "Yes"))
    .select(
        "CDE_ID",
        "Title_ID",
        "Title_Name",
        "film_id",
        "datefull",
        "Quantity",
        T_trans.LocationId
    )
    .groupBy(
        "CDE_ID", "Title_ID", "Title_Name", "film_id", "datefull", T_trans.LocationId
    )
    .agg(F.sum("Quantity").alias("No_of_Tickets"))
    .filter(F.col("No_of_Tickets") > 0)
)

df_movie_with_runtime = (
    df_movie_views
    .join(T_films.select("film_id", "Film_Runtime"), "film_id", "left")
    .withColumn("runtime", F.coalesce(F.col("Film_Runtime"), F.lit(0)))
)

df_movie_with_province = (
    df_movie_with_runtime
    .join(T_locations.select("LocationId", "Province_CD"), "LocationId", "left")
    .filter(F.col("Province_CD").isNotNull())
)

df_distinct_watches = (
    df_movie_with_province
    .join(T_genre, "film_id", "left")
    .select(
        "Province_CD",
        "CDE_ID",
        "film_id",
        "Title_ID",
        "Title_Name",
        "runtime",
        "datefull",
        "genre_hierarchy_1"
    )
    .dropDuplicates(["Province_CD", "CDE_ID", "title_id", "datefull"])
)

# =====================================================================
# STEP E — MOVIE-LEVEL STATS (RETAIN THIS)
# =====================================================================

df_movie_watch_stats = (
    df_distinct_watches
    .groupBy("Province_CD", "Title_ID", "Title_Name")
    .agg(
        F.count("*").alias("Distinct_Watches"),
        F.sum("runtime").alias("Total_Runtime")
    )
)

window_rank_movies = (
    Window.partitionBy("Province_CD")
    .orderBy(F.col("Distinct_Watches").desc())
)

df_top_movies_by_province = (
    df_movie_watch_stats
    .withColumn("Rank_in_Province", F.rank().over(window_rank_movies))
    .filter(F.col("Rank_in_Province") <= 5)
    .select(
        "Province_CD",
        "Rank_in_Province",
        "Title_Name",
        "Distinct_Watches"
    )
    .orderBy("Province_CD", "Rank_in_Province")
)

# =====================================================================
# STEP F — GENRE-LEVEL STATS (NEW)
# =====================================================================

df_genre_watch_stats = (
    df_distinct_watches
    .filter(F.col("genre_hierarchy_1").isNotNull())
    .groupBy("Province_CD", "genre_hierarchy_1")
    .agg(
        F.count("*").alias("Distinct_Watches"),
        F.sum("runtime").alias("Total_Runtime")
    )
)

window_rank_genres = (
    Window.partitionBy("Province_CD")
    .orderBy(F.col("Distinct_Watches").desc())
)

df_top_genres_by_province = (
    df_genre_watch_stats
    .withColumn("Rank_in_Province", F.rank().over(window_rank_genres))
    .filter(F.col("Rank_in_Province") <= 5)
    .select(
        "Province_CD",
        "Rank_in_Province",
        F.col("genre_hierarchy_1").alias("Genre"),
        "Distinct_Watches"
    )
    .orderBy("Province_CD", "Rank_in_Province")
)

# =====================================================================
# FINAL OUTPUTS
# =====================================================================

df_top_movies_by_province      # Top 5 movies per province
df_top_genres_by_province      # Top 5 genres per province


In [0]:
df_top_movies_by_province.display()
df_top_genres_by_province.display()

In [0]:
df_theater_ranking = (
    T_trans
    .join(T_date.select("dateid", "datefull"), T_trans.VisitDateId == T_date.dateid, "left")
    .join(T_locations.select("LocationId", "Location_Name","Province_CD"), T_trans.LocationId == T_locations.LocationID, "left")
    .filter((F.col("TicketFLAG") == 'Yes') & (F.col("CineClubFLAG")== 'Yes'))
    .filter(F.col("datefull").between("2025-01-01", yesterday))
    .groupBy("Province_CD", "Location_Name")
    .agg(
        F.sum("Quantity").alias("No_of_Tickets"),
    )
    .orderBy("No_of_Tickets", ascending=False)
    
)

In [0]:
df_theater_ranking.display()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# =====================================================================
# STEP A — Build movie views with correct LocationId (refund-safe)
# =====================================================================
df_movie_views = (
    df_all_trans
    .join(T_films, "film_id", "left")
    .filter(
        (F.col("TicketFLAG") == "Yes") &
        (F.col("CineClubFLAG") == "Yes")
    )
    .select(
        "CDE_ID",
        "Title_ID",
        "Title_Name",
        "film_id",
        "datefull",
        "Quantity",
        T_trans.LocationId       # <-- IMPORTANT FIX
    )
    .groupBy(
        "CDE_ID", "Title_ID", "Title_Name", "film_id", "datefull",  T_trans.LocationId
    )
    .agg(F.sum("Quantity").alias("No_of_Tickets"))
    .filter(F.col("No_of_Tickets") > 0)
)

# =====================================================================
# STEP B — Attach runtime
# =====================================================================
df_movie_with_runtime = (
    df_movie_views
    .join(
        T_films.select("film_id", "Film_Runtime"),
        "film_id",
        "left"
    )
    .withColumn("runtime", F.coalesce(F.col("Film_Runtime"), F.lit(0)))
)

# =====================================================================
# STEP C — Attach Province (correct join)
# =====================================================================
df_movie_with_province = (
    df_movie_with_runtime
    .join(
        T_locations.select("LocationId", "Province_CD"),
        "LocationId",
        "left"
    )
    .filter(F.col("Province_CD").isNotNull())
)

# =====================================================================
# STEP D — DISTINCT WATCHES (visit-level)
# =====================================================================
df_distinct_watches = (
    df_movie_with_province
    .select(
        "Province_CD",
        "CDE_ID",
        "film_id",
        "Title_ID",
        "Title_Name",
        "runtime",
        "datefull"
    )
    # ❗ Must dedupe on SAME KEYS used for distinct watches
    .dropDuplicates(["Province_CD", "CDE_ID", "Title_ID", "datefull"])
)

# =====================================================================
# STEP E — Count DISTINCT WATCHES + Total Runtime
# =====================================================================
df_movie_watch_stats = (
    df_distinct_watches
    .groupBy("Province_CD", "Title_ID", "Title_Name")
    .agg(
        F.count("*").alias("Distinct_Watches"),
        F.sum("runtime").alias("Total_Runtime")
    )
)

# =====================================================================
# STEP F — Rank inside each province
# =====================================================================
window_rank = (
    Window.partitionBy("Province_CD")
    .orderBy(F.col("Distinct_Watches").desc())
)

df_ranked_movies = (
    df_movie_watch_stats
    .withColumn("Rank_in_Province", F.row_number().over(window_rank))
)

# =====================================================================
# STEP G — Pick top 3 movies per province
# =====================================================================
df_top_movies_by_province = (
    df_ranked_movies
    .filter(F.col("Rank_in_Province") <= 3)
    .orderBy("Province_CD", "Rank_in_Province")
)

df_top_movies_by_province


In [0]:
df_top_movies_by_province.display()

In [0]:
df_top3_movies_per_province.display()

In [0]:
df_total_minutes_per_province = (
    df_unique_visits
    .groupBy("Province_CD")
    .agg(
        F.sum("runtime").alias("Total_Minutes_Watched_Province")
    )
    .orderBy("Province_CD")
)

df_total_minutes_per_province


In [0]:
df_total_minutes_per_province.display()

In [0]:
# ---------------------------------------------------
# Step 1: Join df_pre_conc with Province
# ---------------------------------------------------
df_conc_with_prov = (
    df_pre_conc
    .join(T_locations.withColumnRenamed("Province_CD", "Prov"), "LocationId", "left")
)

# ---------------------------------------------------
# Step 2: Compute popcorn + drink quantities (your logic)
# ---------------------------------------------------
df_pop_drink = (
    df_conc_with_prov
    .withColumn(
        "Large_Popcorn",
        F.when(F.col("Item_Description").like("%POPCORN%LARGE%"), F.col("Quantity"))
         .when(F.col("Item_Description").like("%LARGE%POPCORN%"), F.col("Quantity"))
         .when(F.col("Item_Description").like("%VHO%CORN%VIP%POPCORN%LARGE%"), F.col("Quantity"))
         .otherwise(0)
    )
    .withColumn(
        "Regular_Popcorn",
        F.when(F.col("Item_Description").like("%POPCORN%REGULAR%"), F.col("Quantity"))
         .when(F.col("Item_Description").like("%REGULAR%POPCORN%"), F.col("Quantity"))
         .when(F.col("Item_Description").like("%VHO%CORN%VIP%POPCORN%REGULAR%"), F.col("Quantity"))
         .otherwise(0)
    )
    .withColumn(
        "Small_Popcorn",
        F.when(F.col("Item_Description").like("%POPCORN%SMALL%"), F.col("Quantity"))
         .when(F.col("Item_Description").like("%SMALL%POPCORN%"), F.col("Quantity"))
         .when(F.col("Item_Description").like("%VHO%CORN%VIP%POPCORN%SMALL%"), F.col("Quantity"))
         .otherwise(0)
    )
    .withColumn(
        "Upsize_S_to_R_Popcorn",
        F.when(F.col("Item_Description") == "VHO CORN - UPSIZE TO REGULAR CORN", F.col("Quantity")).otherwise(0)
    )
    .withColumn(
        "Upsize_R_to_L_Popcorn",
        F.when(F.col("Item_Description") == "VHO CORN - UPSIZE TO LARGE CORN", F.col("Quantity")).otherwise(0)
    )
    .withColumn(
        "Upsize_S_to_L_Popcorn",
        F.when(F.col("Item_Description") == "VHO CORN - UPSIZE SMALL TO LARGE CORN", F.col("Quantity")).otherwise(0)
    )
    .withColumn(
        "Large_Fountain",
        F.when(F.col("Item_Description").like("%VHO%DRINK%CUP%LARGE%"), F.col("Quantity"))
         .when(F.col("Item_Description").like("%VHO%DRINK%VIP%DRINK%LARGE%"), F.col("Quantity"))
         .otherwise(0)
    )
    .withColumn(
        "Regular_Fountain",
        F.when(F.col("Item_Description").like("%VHO%DRINK%CUP%REGULAR%"), F.col("Quantity"))
         .when(F.col("Item_Description").like("%VHO%DRINK%VIP%DRINK%REGULAR%"), F.col("Quantity"))
         .otherwise(0)
    )
    .withColumn(
        "Small_Fountain",
        F.when(F.col("Item_Description").like("%VHO%DRINK%CUP%SMALL%"), F.col("Quantity")).otherwise(0)
    )
    .withColumn(
        "Kiddie_Fountain",
        F.when(F.col("Item_Description").like("%VHO%DRINK%CUP%KIDDIE%"), F.col("Quantity")).otherwise(0)
    )
    .withColumn(
        "Upsize_S_XL",
        F.when(F.col("Item_Description").like("%UPSIZE SMALL FOUNTAIN%XL%"), F.col("Quantity")).otherwise(0)
    )
    .withColumn(
        "Upsize_R_XL",
        F.when(F.col("Item_Description").like("%UPSIZE REGULAR FOUNTAIN%XL%"), F.col("Quantity")).otherwise(0)
    )
    .withColumn(
        "Upsize_L_XL",
        F.when(F.col("Item_Description").like("%UPSIZE LARGE FOUNTAIN%XL%"), F.col("Quantity")).otherwise(0)
    )
    # ---------------------------------------------------
    # Step 3: Total Kernels + Total Drink ML
    # ---------------------------------------------------
    .withColumn(
        "Total_Kernels",
        (
            (F.col("Small_Popcorn") * 300) +
            (F.col("Regular_Popcorn") * 400) +
            (F.col("Large_Popcorn") * 500) +
            (F.col("Upsize_S_to_R_Popcorn") * 400) +
            (F.col("Upsize_R_to_L_Popcorn") * 500) +
            (F.col("Upsize_S_to_L_Popcorn") * 500)
        )
    )
    .withColumn(
        "Total_Drink_ml",
        (
            (F.col("Small_Fountain") * 250) +
            (F.col("Regular_Fountain") * 350) +
            (F.col("Large_Fountain") * 500) +
            (F.col("Upsize_S_XL") * 500) +
            (F.col("Upsize_R_XL") * 500) +
            (F.col("Upsize_L_XL") * 500)
        )
    )
)

# ---------------------------------------------------
# Step 4: Aggregate totals BY PROVINCE
# ---------------------------------------------------
df_kernels_drinks_by_province = (
    df_pop_drink
    .groupBy("Prov")
    .agg(
        F.sum("Total_Kernels").alias("Total_Kernels_Province"),
        F.sum("Total_Drink_ml").alias("Total_Drink_ml_Province")
    )
    .orderBy("Prov")
)

df_kernels_drinks_by_city = (
    df_pop_drink
    .groupBy("Prov", "City")
    .agg(
        F.sum("Total_Kernels").alias("Total_Kernels_City"),
        F.sum("Total_Drink_ml").alias("Total_Drink_ml_City")
    )
    .orderBy("Prov")
)

In [0]:
df_kernels_drinks_by_city.display()

## Favorite Snack

In [0]:
df_candy = (
    df_pre_conc_test
    .join(T_items, "Item_ID", "left")
    .join(T_locations.withColumnRenamed("Province_CD", "Prov"), "LocationId", "left")
    .groupBy("Prov", "City")
    .agg(
        F.sum(
            F.when(F.col("Item_Class_Description1").like("%VHO-CANDY%"), F.col("Quantity"))
             .otherwise(0)
        ).alias("Candy_Count")
    )
)


In [0]:
df_candy.display()

In [0]:
df_movie_views = (
    df_all_trans
    .join(T_films, df_all_trans.film_id == T_films.film_id, "left")
    .filter(F.col("TicketFLAG") == "Yes")
    .select("CDE_ID", "Title_ID", "Title_Name", df_all_trans.film_id, "datefull","Quantity","Province_CD")
    .groupBy("CDE_ID", "Title_ID", "Title_Name", df_all_trans.film_id, "datefull", "Province_CD")
    .agg(
        F.sum(F.col("Quantity")).alias("No_of_Tickets")
    )
    .filter(F.col("No_of_Tickets")>0)
    .dropDuplicates()           # ensures 1 watch per movie per day
)

In [0]:
from pyspark.sql import functions as F

# -------------------------------------------------------
# 0. Extract Top 10 Movies (title_id, Movie_No, Movie_Name)
# -------------------------------------------------------
df_top10 = df_top10_movies_member_based.select("title_id", "Movie_No", "Movie_Name")

# -------------------------------------------------------
# 1. Movie views joined with province + filter CineClub members
# -------------------------------------------------------
df_views_cc = (
    df_movie_views
    .join(df_cineclub_members, "CDE_ID", "inner")              # CC members only
)

# -------------------------------------------------------
# 2. JOIN top 10 movies → keep only those movies
# -------------------------------------------------------
df_top10_views = (
    df_views_cc
    .join(df_top10, "title_id", "inner")
)

# -------------------------------------------------------
# 3. Distinct watch count per province for each movie
# -------------------------------------------------------
df_top10_watch_by_prov = (
    df_top10_views
    .groupBy("Movie_No", "Movie_Name", "Province_CD")
    .agg(F.countDistinct("CDE_ID").alias("Distinct_CC_Watchers"))
    .orderBy("Movie_No", "Province_CD")
)


In [0]:
df_top10_watch_by_prov.display()

In [0]:
# -------------------------------------------------------
# 1. Join CineClub members with any location to determine province
# -------------------------------------------------------
df_member_province = (
    df_cineclub_members
    .join(df_all_trans.select("CDE_ID", "Province_CD"), "CDE_ID", "left")
    .select("CDE_ID", "Province_CD")
    .distinct()
)

# -------------------------------------------------------
# 2. Count members by province
# -------------------------------------------------------
df_cc_member_count_by_prov = (
    df_member_province
    .groupBy("Province_CD")
    .agg(F.countDistinct("CDE_ID").alias("Total_CC_Members"))
    .orderBy("Province_CD")
)


In [0]:
df_cc_member_count_by_prov.display()
